# 📘 Notebook 09: Regularisation, Dropout, Weight Decay and Data Augmentation

### Course: *From Defaults to Decisions: Hyperparameters, Explained by Experiment*
**Instructor:** Ioannis Tsioulis

---

> 💡 **This notebook is already run.** Every number and chart below was produced by the code above it, and each result is followed by a comment that explains what it shows. You can read it without running anything, or choose **Runtime → Run all** in Colab.
>
> ⏱️ **Training time and limitations.** About **twenty-five minutes** on an ordinary processor. To make overfitting strong and visible, we use a large network (two layers of 1,024 neurons) on only **2,000** training images, trained for 60 epochs without early stopping. With more data or early stopping, the gains from regularisation are smaller, but the mechanisms are the same.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Recognise **overfitting** as a growing gap between training and validation accuracy
- Explain and apply **dropout**, **weight decay** and **data augmentation**
- Sweep the strength of each and find where it stops helping and starts hurting
- Combine regularisers, and compare them with simply getting **more data**

> **Prerequisites:** [Notebook 02: Epochs and Early Stopping](https://colab.research.google.com/github/giannis00/giannis00.github.io/blob/main/assets/jupyter/from-defaults-to-decisions/02_epochs_and_early_stopping.ipynb) and [Notebook 07: Optimisers](https://colab.research.google.com/github/giannis00/giannis00.github.io/blob/main/assets/jupyter/from-defaults-to-decisions/07_optimisers.ipynb) for AdamW.
>
> 🔭 **Why this notebook matters:** most real projects have less data than they would like. Regularisation is how a large model is kept from memorising a small dataset, and each method has a strength that must be tuned like any other hyperparameter.

## 1. The papers

> 📄 Dropout: N. Srivastava, G. Hinton, A. Krizhevsky, I. Sutskever and R. Salakhutdinov, [*Dropout: A Simple Way to Prevent Neural Networks from Overfitting*](https://jmlr.org/papers/v15/srivastava14a.html), JMLR 15, 2014.
>
> 📄 Weight decay in Adam: I. Loshchilov and F. Hutter, [*Decoupled Weight Decay Regularization*](https://arxiv.org/abs/1711.05101), ICLR 2019.

### Intuition first
- **Dropout:** during training, switch off each neuron at random with probability $p$. No neuron can rely on a particular partner, so the network learns redundant, more general features. At test time all neurons are used.
- **Weight decay:** pull every weight a little towards zero at each step. Large weights let a network draw very sharp, wiggly boundaries around individual training images; small weights keep it smooth.
- **Data augmentation:** show slightly changed copies of the training images (shifted, flipped). The network sees more variety, and learns that these changes do not alter the label.

In [1]:
# Shared setup: the same data, model builder and training loop in every notebook of this course.
import warnings
warnings.filterwarnings("ignore")

import gzip, time, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(4)
plt.rcParams["figure.dpi"] = 80
pd.set_option("display.precision", 3)

BASE = "https://github.com/zalandoresearch/fashion-mnist/raw/master/data/fashion/"
def read_idx(name, header):
    return np.frombuffer(gzip.decompress(urllib.request.urlopen(BASE + name).read()), dtype=np.uint8, offset=header)

images = torch.tensor(read_idx("train-images-idx3-ubyte.gz", 16).reshape(-1, 784), dtype=torch.float32) / 255
labels = torch.tensor(read_idx("train-labels-idx1-ubyte.gz", 8), dtype=torch.long)
X_test = torch.tensor(read_idx("t10k-images-idx3-ubyte.gz", 16).reshape(-1, 784), dtype=torch.float32) / 255
y_test = torch.tensor(read_idx("t10k-labels-idx1-ubyte.gz", 8), dtype=torch.long)
mean, std = images.mean(), images.std()
images, X_test = (images - mean) / std, (X_test - mean) / std

order = torch.randperm(len(images), generator=torch.Generator().manual_seed(0))
X_train, y_train = images[order[:10000]], labels[order[:10000]]       # 10,000 images to learn from
X_val, y_val = images[order[10000:15000]], labels[order[10000:15000]] # 5,000 to choose hyperparameters
print(f"train {len(X_train):,}   validation {len(X_val):,}   test {len(X_test):,}")

def make_mlp(hidden=(128,), activation=nn.ReLU, dropout=0.0, batchnorm=False):
    layers, width = [], 784
    for h in hidden:
        layers.append(nn.Linear(width, h))
        if batchnorm:
            layers.append(nn.BatchNorm1d(h))
        layers.append(activation())
        if dropout:
            layers.append(nn.Dropout(dropout))
        width = h
    layers.append(nn.Linear(width, 10))
    return nn.Sequential(*layers)

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

def fit(model, epochs=20, lr=1e-3, batch=128, optimizer="adam", weight_decay=0.0, scheduler=None, seed=0, X=None, y=None):
    """Train and return a history with one row per epoch."""
    X = X_train if X is None else X
    y = y_train if y is None else y
    torch.manual_seed(seed)
    opts = {"sgd": lambda p: torch.optim.SGD(p, lr=lr, weight_decay=weight_decay),
            "momentum": lambda p: torch.optim.SGD(p, lr=lr, momentum=0.9, weight_decay=weight_decay),
            "rmsprop": lambda p: torch.optim.RMSprop(p, lr=lr, weight_decay=weight_decay),
            "adam": lambda p: torch.optim.Adam(p, lr=lr, weight_decay=weight_decay),
            "adamw": lambda p: torch.optim.AdamW(p, lr=lr, weight_decay=weight_decay)}
    opt = opts[optimizer](model.parameters())
    steps_per_epoch = (len(X) + batch - 1) // batch
    sched = scheduler(opt, epochs * steps_per_epoch) if scheduler else None
    rows, start = [], time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(X))
        total = 0.0
        for i in range(0, len(X), batch):
            idx = perm[i:i + batch]
            loss = F.cross_entropy(model(X[idx]), y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            if sched:
                sched.step()
            total += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(X_val), y_val).item()
        rows.append({"epoch": epoch, "train loss": total / len(X), "val loss": val_loss,
                     "train acc": accuracy(model, X, y), "val acc": accuracy(model, X_val, y_val)})
    history = pd.DataFrame(rows).set_index("epoch")
    history.attrs["seconds"] = time.time() - start
    return history

train 10,000   validation 5,000   test 10,000


In [2]:
X_small, y_small = X_train[:2000], y_train[:2000]
EPOCHS = 60
def run(model, **kw):
    h = fit(model, epochs=EPOCHS, X=kw.pop("X", X_small), y=kw.pop("y", y_small), **kw)
    return {"train acc": h["train acc"].iloc[-1], "val acc": h["val acc"].iloc[-1], "best val acc": h["val acc"].max(),
            "gap": h["train acc"].iloc[-1] - h["val acc"].iloc[-1]}

baseline = run(make_mlp(hidden=(1024, 1024)))
print({k: round(v, 4) for k, v in baseline.items()})

{'train acc': 1.0, 'val acc': 0.8286, 'best val acc': 0.8342, 'gap': 0.1714}


**What we see.** The baseline overfits as completely as a network can: **100%** training accuracy against **82.9%** validation, a gap of **17 points**. Two layers of 1,024 neurons have about 1.8 million weights for 2,000 images, enough to memorise every one. The best validation accuracy during training was 83.4%, only half a point above the last epoch: as in Notebook 02, overfitting shows much more in the loss than in the accuracy.

## 2. Dropout

In [3]:
drop_rows = {p: run(make_mlp(hidden=(1024, 1024), dropout=p)) for p in [0.0, 0.2, 0.4, 0.6, 0.8]}
pd.DataFrame(drop_rows).T.rename_axis("dropout probability").round(4)

,train acc,val acc,best val acc,gap
dropout probability,,,,
0.0,1.000,0.836,0.838,0.164
0.2,0.997,0.823,0.832,0.174
0.4,0.991,0.827,0.837,0.164
0.6,0.973,0.836,0.839,0.137
0.8,0.912,0.830,0.832,0.083


**What we see.** Dropout narrows the gap, but mostly by **lowering the training accuracy**, not by raising the validation accuracy:
- From p = 0 to 0.6, validation accuracy moves between **82.3% and 83.6%**, differences within the seed-to-seed noise of Notebook 01. Training accuracy falls from 100% to 97.3%.
- At **p = 0.8** the gap halves (**8.3** points), yet validation accuracy is no better (83.0%): the network can no longer even fit its training images well.

This is an honest and common outcome: a smaller gap is **not** the goal in itself; higher validation accuracy is. Here the network already lost little validation accuracy to overfitting (Section 1), so dropout had little to recover.

## 3. Weight decay (with AdamW)

In [4]:
wd_rows = {wd: run(make_mlp(hidden=(1024, 1024)), optimizer="adamw", weight_decay=wd) for wd in [0.0, 0.01, 0.1, 0.5, 2.0]}
pd.DataFrame(wd_rows).T.rename_axis("weight decay").round(4)

,train acc,val acc,best val acc,gap
weight decay,,,,
0.00,0.993,0.823,0.833,0.169
0.01,1.000,0.838,0.839,0.162
0.10,1.000,0.835,0.836,0.165
0.50,0.985,0.817,0.832,0.167
2.00,0.943,0.797,0.831,0.145


**What we see.** Weight decay behaves the same way. With AdamW, decay of **0.01** and **0.1** gives the best final validation accuracies (**83.8%** and **83.5%**, against 82.3% without decay), a gain of about one point that a few more seeds would be needed to confirm. Strong decay (**0.5** and **2.0**) pulls the weights so hard that the training accuracy drops (to 94.3%) and the final validation accuracy falls to **79.7%**: too much regularisation causes underfitting. Note the "best val acc" column: at its best epoch, every setting reaches about 83.1 to 83.9%. Most of what weight decay did here was keep the **final** epoch close to the best one.

## 4. Data augmentation

For clothes, a small **shift** and a **left-right flip** do not change the class (a shirt flipped is still a shirt). We create a new random version of every batch.

In [5]:
def augment(x):
    imgs = x.view(-1, 1, 28, 28)
    flip = torch.rand(len(imgs)) < 0.5
    imgs = torch.where(flip[:, None, None, None], imgs.flip(3), imgs)
    dx, dy = np.random.randint(-2, 3, size=2)
    return torch.roll(imgs, shifts=(int(dy), int(dx)), dims=(2, 3)).reshape(len(x), 784)

def fit_augmented(model, epochs=EPOCHS, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(X_small))
        for i in range(0, len(X_small), 128):
            idx = perm[i:i + 128]
            loss = F.cross_entropy(model(augment(X_small[idx])), y_small[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return {"train acc": accuracy(model, X_small, y_small), "val acc": accuracy(model, X_val, y_val)}

aug = fit_augmented(make_mlp(hidden=(1024, 1024)))
print("with augmentation:", {k: round(v, 4) for k, v in aug.items()}, "  gap", round(aug["train acc"] - aug["val acc"], 4))

with augmentation: {'train acc': 0.874, 'val acc': 0.7868}   gap 0.0872


**What we see.** Augmentation narrowed the gap the most (to **8.7** points), but in the wrong way: training accuracy fell to 87.4% and validation accuracy **fell** to **78.7%**, four points below the baseline. Two things went against it here. First, a fully connected network sees each pixel position as a separate input, so a shifted image is to it almost a different image; learning all shifted and flipped versions of 2,000 images is a much harder task, and 60 epochs were not enough (training accuracy is still far from 100%). Second, the validation images are **not** shifted or flipped, so the extra variety the network spent its capacity on never appears at test time. Augmentation pays off when the changes match the variation in real data, and with convolutional networks, which handle shifts by design.

## 5. Combining them, and the best regulariser of all

In [6]:
combined = run(make_mlp(hidden=(1024, 1024), dropout=0.4), optimizer="adamw", weight_decay=0.1)
more_data = run(make_mlp(hidden=(1024, 1024)), X=X_train, y=y_train)
table = pd.DataFrame({"no regularisation (2,000 images)": baseline, "dropout 0.4 + weight decay 0.1 (2,000 images)": combined,
                      "no regularisation, 10,000 images": more_data}).T
table.round(4)

,train acc,val acc,best val acc,gap
"no regularisation (2,000 images)",1.000,0.829,0.834,0.171
"dropout 0.4 + weight decay 0.1 (2,000 images)",0.995,0.825,0.838,0.170
"no regularisation, 10,000 images",0.977,0.850,0.873,0.126


**What we see.** Dropout and weight decay together give **83.8%** best validation accuracy, against 83.4% without them: a small gain, within noise. Training the **same** unregularised network on **10,000** images instead of 2,000 gives **87.3%**, about four points more than any regulariser achieved, with a smaller gap. The lesson of this notebook in one line: regularisers are fine-tuning, while **more real data** is the strongest regulariser there is.

## ⚠️ Common pitfalls

- **Forgetting `model.eval()`** when measuring: dropout stays on and accuracy looks worse and random.
- **Too much regularisation:** the model underfits; training accuracy falls towards validation accuracy and both are low.
- **Augmentations that change the label:** flipping a 6 into a 9, or a left arrow into a right one.
- **Weight decay with Adam instead of AdamW.**
- **Regularising a model that underfits.** First make it fit the training data; then regularise.

## ✏️ Exercises

### Exercise 1 (Dropout on more data)
Repeat the dropout sweep (0, 0.4) on all 10,000 training images. Is the gain from dropout larger or smaller than with 2,000?

In [7]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for p in [0.0, 0.4]:
    r = run(make_mlp(hidden=(1024, 1024), dropout=p), X=X_train, y=y_train)
    print(f"dropout {p}: val acc {r['val acc']:.4f}, gap {r['gap']:.4f}")
```

*A result that runs against the usual expectation: in our run dropout 0.4 helped **more** with 10,000 images (85.9% to 86.9%, gap from 13.1 to 10.5 points) than with 2,000, where it made no measurable difference. A likely reason: with 2,000 images and 60 epochs the network memorises the training set either way, while with more data dropout has real patterns to protect. The common advice that "regularisation matters most when data is scarce" is a tendency, not a law; it has to be measured, like everything else in this course.*
</details>

### Exercise 2 (Which augmentation helps?)
Run `fit_augmented` with only the shift and with only the flip (edit `augment`). Which one does more here?

In [8]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def shift_only(x):
    imgs = x.view(-1, 1, 28, 28)
    dx, dy = np.random.randint(-2, 3, size=2)
    return torch.roll(imgs, shifts=(int(dy), int(dx)), dims=(2, 3)).reshape(len(x), 784)
def flip_only(x):
    imgs = x.view(-1, 1, 28, 28)
    flip = torch.rand(len(imgs)) < 0.5
    return torch.where(flip[:, None, None, None], imgs.flip(3), imgs).reshape(len(x), 784)
for name, f in [("shift only", shift_only), ("flip only", flip_only)]:
    augment = f
    print(name, {k: round(v, 4) for k, v in fit_augmented(make_mlp(hidden=(1024, 1024))).items()})
```

*In our run neither helped on its own: shift only reached 81.3% and flip only 82.1%, both below the 82.9% of the baseline, and the shift lowered the training accuracy most (to 90%). This confirms Section 4: for a fully connected network, a shifted image is almost a different input, and the validation images are neither shifted nor flipped. The best augmentation is the one that covers the variation real data will contain; for Fashion-MNIST, whose images are already centred, there is little such variation to cover.*
</details>

## 🔑 Key takeaways

- **Overfitting** shows as a large gap between training and validation accuracy.
- **Dropout**, **weight decay** (with AdamW) and **data augmentation** each narrow the gap, but a smaller gap is not the goal: judge them by **validation** accuracy. Each has a strength to tune, and too much causes underfitting.
- **Augmentation** helps only when the changes match real variation and the model can absorb them; here, with a fully connected network, it hurt.
- Regularisers can be **combined**, but more real **data** is usually the strongest regulariser of all.

---
**Next:** *Notebook 10: Batch Normalisation and Learning-Rate Schedules*, two tools that make training faster and more forgiving.

---
*© Ioannis Tsioulis. *From Defaults to Decisions: Hyperparameters, Explained by Experiment*. Prepared for educational use.*